In [0]:
#standardization and replacement
#standardization1 - Column Enrichment (Addition of columns)
#create new column with default value
from pyspark.sql.functions import col,lit,upper

standdf1=spark.read.csv(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/staging/custs",inferSchema=True).toDF("custid","fname","lname","age","profession")
standdf2=standdf1.withColumn("source",lit("raw"))
#display(standdf2)

#Standardization2 - Column Uniformity
#profession column Uniformity with upper
standdf3=standdf2.withColumn("profession",upper("profession"))
#display(standdf3.limit(20))

#Standardization3 - Format Standardization
#check id and age column if it contains non integer values
standdf3.where("custid rlike '[a-zA-Z]'").show()
standdf3.where("age rlike '[^0-9]'").show()

standdf3.printSchema()

#replace ten with 10 using replace function 
#regreplace of age by removing - between 4-7
from pyspark.sql.functions import replace,regexp_replace
replacedict={'one':'1','two':'2','three':'3','four':'4','five':'5','six':'6','seven':'7','eight':'8','nine':'9','ten':'10'}
standdf4=standdf3.na.replace(replacedict,["custid"])
#standdf4.where("custid rlike '[a-zA-Z]'").show()
standdf4.where("custid='10'").show()

standdf5=standdf4.withColumn('age',regexp_replace(col('age'),"-",""))
#standdf5.where("age rlike '[^0-9]'").show()
standdf5.where("age=47").show(10,False)

In [0]:
#Standardization4 - Data Type Standardization
standdf5.printSchema()

standdf6=standdf5.withColumn("custid",standdf5['custid'].cast('long'))
standdf6=standdf6.withColumn("age",standdf5['age'].cast('short'))
standdf6.printSchema()

In [0]:
#Standardization5 - Naming Standardization
#rename the existing columns

standdf7=standdf6.withColumnsRenamed({'custid':'CustomerID','fname':'FirstName','lname':'LastName','age':'Age','profession':'Profession','source':'Source'})
display(standdf7.limit(10))


In [0]:
#Standardization6 - Reorder Standadization
#original column order in dataframe
#display(standdf7).limit(10)
#reorder
#display(standdf7.select("Age","Profession","FirstName","LastName","CustomerID","Source").limit(10))

#drop column using na.drop
standdf8=standdf7.na.drop(subset=['profession']) #this will remove only the null value present in the profession column
display(standdf8.count())

standdf9=standdf8.drop("Source")
display(standdf9)


In [0]:
#Passive data munging
#high level
#schema1="custid string,fname string,lname string,age string,profession string"
rawdf1=spark.read.csv(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/BB1 Customers/",inferSchema=True, pathGlobFilter="cust*",recursiveFileLookup=True).toDF("custid","fname","lname","age","profession")
#rawdf1.show(10,False) > Pyspark show dont Uses driver memory and not interactive and Limited formatting. take file data into python list of values in a dataframe.
#display(rawdf1.take(10)) > Databricks take Uses driver memory but more interactive, display function render them in a table and more formatting
'''
rawdf1.printSchema()
print(rawdf1.columns)
print(rawdf1.dtypes)
print(rawdf1.schema)

#count
display(rawdf1.count())

#summary and description
display(rawdf1.describe())
display(rawdf1.summary())
'''
#remove duplicates
from pyspark.sql.functions import col

display(rawdf1.count()) #gives total number of rows in dataframe includes duplicates
display(rawdf1.groupBy("custid").count().orderBy(col("custid").asc()))

#display(rawdf1.distinct().count()) #gives total number of rows in dataframe after removing duplicates entire row
#display(rawdf1.dropDuplicates().count()) #gives total number of rows in dataframe after removing duplicates in a column level
display(rawdf1.dropDuplicates(['custid']).count()) #removes duplicate on specified column 'custid'




In [0]:
#Active data munging
#Take data from two difference source and merge/melt it togather
from pyspark.sql.types import StructType,StructField,StringType
nyschema=StructType([StructField('custid', StringType(), True), StructField('fname', StringType(), True), StructField('lname', StringType(), True), StructField('age', StringType(), True), StructField('profession', StringType(), True)])

nydf1=spark.read.schema(nyschema).csv(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer",pathGlobFilter="*_NY",recursiveFileLookup=True)
display(nydf1)

txschema="custid int,fname string, age int,profession string,lname string"
txdf2=spark.read.schema(txschema).format("csv").options(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer",pathGlobFilter="*_TX",recursiveFileLookup=True).load()
display(txdf2)

ny_txdf=nydf1.unionByName(txdf2,allowMissingColumns=True)
display(ny_txdf)

#implement schema evalution
#step-1: read file as a dataframe
txschema1="custid int,fname string, age int,profession string,lname string,city string"
txdf3=spark.read.schema(txschema1).csv(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/TX/custsmodified_TX1.txt")
display(txdf3)

#step-2: write the above two dataframe's into orc/parquet format
txdf2.write.orc(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/TX/ORC_TX",mode="append")
txdf3.write.orc(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/TX/ORC_TX",mode="append")

#step-3: read the orc data with mergeschema=True to get newly added columns from the source
finaltxdf=spark.read.orc(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/customer/TX/ORC_TX",mergeSchema=True)
display(finaltxdf)

In [0]:
#Rejection Strategy
newschema="id string,fname string,lname string,age string,profession string,corruptedcol string"
df01=spark.read.schema(newschema).csv(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/BB1 Customers/custsmodified",mode="permissive",columnNameOfCorruptRecord="corruptedcol")

print("Original Dataframe Count:",df01.count())
#method-1 using where function
data_without_correupted_col=df01.where("corruptedcol IS NOT NULL")
display(data_without_correupted_col)

#method-2: using drop function
df02=data_without_correupted_col.na.drop(subset=["corruptedcol"])
display(df02)

#multiple columns with null
df03=df02.na.drop(how='any',subset=["age","profession"])
display(df03)

newdf04=df01.na.drop(subset=["profession"])
display(newdf04.count())

#remove duplicate of ID column
newdf05=df01.dropDuplicates(['id'])
display(newdf05.count())

In [0]:
from pyspark.sql.functions import lit,col,initcap,replace,regexp_replace,when,current_date,substr,substring,upper,concat

testschema="Id string,Fname string,Lname string,Age string,Profession string"
df0201=spark.read.schema(testschema).csv(path="/Volumes/telecom_catalog_assign/landing_zone/landing_vol/BB1 Customers/custsmodified")
#display(df0201)
df0202=df0201.withColumn('Source',lit("Retail"))
#display(df0202)
df0203=df0202.withColumn('Profession',initcap('Profession'))
#display(df0203.limit(100))
#df0203.where("Id rlike '[a-zA-Z]'").show()
#df0203.where("Age rlike '[^0-9]'").show()
renameid={'ten':'10'}
df0204=df0203.na.replace(renameid,['Id'])
#display(df0204.filter("Id='10'"))
df0205=df0204.withColumn('Age',regexp_replace('Age','[^0-9]',''))
#display(df0205)
df0206=df0205.filter("Id is not null and Age is not null and Profession is not null")
#display(df0206.count())
df0207=df0206.withColumn("Lname",when(col("Lname").isNull(),lit("Not Applicable")).otherwise(col("Lname")))
#display(df0207)
#df0207.printSchema()
standarddf1=df0207.withColumn('Id',col('Id').cast("long")).withColumn('Age',df0207.Age.cast("short"))
#standarddf1.printSchema()
df0208=standarddf1.withColumnRenamed('Id','CustID')
#df0208=standarddf1.withColumnRenamed({'Id':'CustID','Fname':'FirstName','Lname':'LastName','Age':'Age','Profession':'Profession','Source':'Source'}) >> THIS IS NOT WORKING
#display(df0208)

file_name="custmodified_04/01/2026.csv"
deriveddate=file_name.split("_")[1].split(".")[0]
#df0401=df0208.withColumn('Derived_Date',lit(deriveddate)).withColumn('Load_Date',current_date())
#df0401=df0208.withColumns({'Derived_Date':lit(deriveddate),'Load_Date':current_date()})
#display(df0401)

#df0402=df0208.select('*',lit(deriveddate).alias('Derived_Date'),current_date().alias('Load_Date')) #dsl function
#display(df0402)

df0403=df0208.selectExpr('*',f"'{deriveddate}' as Derived_Date","current_date() as Load_Date") #dsl + SQL expression
#display(df0403)

df0404=df0403.withColumn('Prof_Flag',upper(substring('Profession',1,2)))
#display(df0404)

df0405=df0404.withColumn('Profession',concat('Profession',lit('-'),'Prof_Flag'))
df0406=df0405.drop('Prof_Flag')
#display(df0406)

my_name='VivekBharathi'
my_select=f"'{my_name}' as Aspirant"
df0407=df0406.selectExpr('*',my_select)
display(df0407)


In [0]:
def ageCalculation(value):
    if value<18:
        return "Minor"
    elif value>=18 and value<60:
        return "Adult"
    else:
        return "Senior Citizen"

from pyspark.sql.functions import udf
dataframe_udf=udf(ageCalculation)
df0408=df0407.withColumn('Age_Group',dataframe_udf(col('Age'))) #df0407['Age'] or col('Age')
#display(df0408.filter(("Age_Group like 'Adult'") and (df0408.Age < 30)).count())
display(df0408)
#df0408.where(df0408.Age > 20).explain()

In [0]:
#data curation/processing (pre wrangling stage)
#select (dsl)
#df0601=df0408.select('CustID',col('Fname').alias('FirstName'),col('Lname').alias('LastName'),'Age','Profession','Source','Derived_Date','Load_Date')
#or using selectexpr (sql)
df0601=df0408.selectExpr('CustID','Fname as FirstName','Lname as LastName','Age','Profession','Source','Derived_Date','Load_Date')
#display(df0601)

#filter with condition
#where and filter both are literaly same. Filter will be used in DSL, Where while using SQL. still no difference

#df0601.filter((col('Age')>40) & (col('Age')<50)).show()
#df0601.where((col('Age')>40) & (col('Age')<50)).show()

#df0601.filter('Age>40 and Age<50').show(5) #using sql in filter or where

#derived column
'''
df0601.select('*',when(col("Age")<18,"Tean")
              .when((col("Age")>=18) & (col("Age")<60),"Adult")
              .when(col("Age")>=60,"Senior Citizen")
              .alias("Age_Group")).show(10) #suggested instead of UDF for performance
'''
df0602 = df0601.selectExpr(
    "*",
    """
    case 
        when Age is null then 'Unknown'
        when Age < 18 then 'Teen'
        when Age >= 18 and Age < 60 then 'Adult'
        when Age >= 60 then 'Senior Citizen'
        else 'Not Applicable'
    end as Age_Category
    """
)
#display(df0602)

from pyspark.sql.functions import datediff,day,month,year,to_date,dayname,monthname

df0603=df0602.withColumn('Derived_Date',to_date('Derived_Date','MM/dd/yyyy'))

df0603.select("*",datediff('Derived_Date','Load_Date').alias('Date_Diff'),
                dayname('Load_Date').alias('Day'),
                year('Load_Date').alias('Year'),
                monthname('Load_Date').alias('Month'),
              ).show()



In [0]:
#What is the total number of customers we have in each profession?
#display(df0603.groupBy('Profession').count())

#Multiple Aggregation with one grouping - What is the total number of customers,average age of those customers we have in each profession?
#display(df0603.groupBy('Profession').avg('Age').withColumnRenamed('avg(Age)','Avg_Age'))

#To calculate multiple aggregation, we need to use a function called agg function
from pyspark.sql.functions import count,avg,round,min,max
display(df0603.groupBy('Profession').agg(count('CustID').alias('Total'),
                                         round(avg('Age'),0).alias('Avg_Age'),
                                         min("Age").alias('Min_Age'),
                                         max("Age").alias('Max_Age')).orderBy('Profession'))


In [0]:
df0603.limit(20).show(5)